# Flatfox – Wohnungsdaten abrufen und bereinigen

**Ziel:** Mietwohnungen aus Zürich und Umgebung über die Flatfox-API abrufen, bereinigen und als CSV sowie SQLite-Datenbank für das Gruppenprojekt speichern.

Dieses Notebook ist die bereinigte Fassung der ursprünglichen Colab-Versuche. Die Schritte sind so sortiert, dass sie von oben nach unten ausgeführt werden können.

**Hinweis:** Die API liefert aktuelle Inserate. Deshalb kann sich die Anzahl der Wohnungen bei einem erneuten Abruf verändern. Die Anfragen sollten in einem mit Flatfox vereinbaren Umfang ausgeführt werden. Code mit Unterstützung von ChatGPT strukturiert; Verwendung im AI-Log der Projektgruppe dokumentieren.

## 1. Bibliotheken laden
`requests` fragt die API ab. `pandas` verwaltet Tabellen. `time` setzt Pausen. `sqlite3` erstellt die Datenbank.

In [ ]:
import requests
import pandas as pd
import time
import sqlite3
import os

os.makedirs("data", exist_ok=True)  # Ordner für die Ergebnisse erstellen

## 2. Wohnungs-IDs aus fünf Gebieten abrufen
Die ursprüngliche Suchregion ist in fünf kleinere Gebiete aufgeteilt, weil Flatfox höchstens 1.000 Pins pro Anfrage liefert. Wir entfernen doppelte Einträge anhand der Wohnungs-ID `pk`.

In [ ]:
gebiete = [
    {"west": 8.35, "east": 8.45, "south": 47.28, "north": 47.50},
    {"west": 8.45, "east": 8.55, "south": 47.28, "north": 47.39},
    {"west": 8.45, "east": 8.55, "south": 47.39, "north": 47.50},
    {"west": 8.55, "east": 8.65, "south": 47.28, "north": 47.50},
    {"west": 8.65, "east": 8.75, "south": 47.28, "north": 47.50},
]

alle_pins_liste = []
url = "https://flatfox.ch/api/v1/pin/"

for nummer, gebiet in enumerate(gebiete, start=1):
    params = {
        "north": gebiet["north"],
        "south": gebiet["south"],
        "east": gebiet["east"],
        "west": gebiet["west"],
        "offer_type": "RENT",
        "object_category": "APARTMENT",
        "max_count": 1000,
    }
    antwort = requests.get(url, params=params, timeout=30)
    antwort.raise_for_status()  # Stoppt bei Serverfehlern
    pins = antwort.json()
    alle_pins_liste.extend(pins)  # Neue Einträge an die Liste anhängen
    print("Gebiet", nummer, ":", len(pins), "Einträge")
    if len(pins) == 1000:
        print("Hinweis: Gebiet", nummer, "erreicht das Limit. Eventuell kleiner aufteilen.")

alle_pins = pd.DataFrame(alle_pins_liste)
alle_pins = alle_pins.drop_duplicates(subset="pk")
alle_pins.to_csv("data/flatfox_pins.csv", index=False)
print("Unterschiedliche IDs:", len(alle_pins))

## 3. Details zu jeder Wohnungs-ID abrufen
Die `pin`-API enthält nur wenige Felder. Über `public-listing/<ID>/` erhalten wir die vollständigen Wohnungsdaten. Mit einer Pause zwischen den Anfragen schonen wir den Server. Bei Statuscode `429` hören wir auf.

**Achtung:** Dieser Schritt ruft viele einzelne URLs ab. Bei Unterbrechung sind bereits abgefragte Einträge zunächst nur im Speicher; nach einem vollständigen Durchlauf speichern wir die Rohdaten.

In [ ]:
details = []

for pk in alle_pins["pk"]:
    url = "https://flatfox.ch/api/v1/public-listing/" + str(pk) + "/"
    antwort = requests.get(url, timeout=30)

    if antwort.status_code == 200:
        details.append(antwort.json())
    elif antwort.status_code == 429:
        print("Zu viele Anfragen (429): Download gestoppt.")
        break
    else:
        print("Inserat", pk, "nicht geladen, Statuscode:", antwort.status_code)

    time.sleep(0.5)

rohdaten = pd.DataFrame(details)
rohdaten.to_csv("data/flatfox_rohdaten.csv", index=False)
print("Heruntergeladene Details:", len(rohdaten))

## 4. Wohnungen bereinigen
Wir behalten reguläre, unmöblierte und unbefristete Mietwohnungen mit monatlichem Mietpreis. Fehlende Bruttomiete wird – sofern beide Werte existieren – aus Nettomiete und Nebenkosten berechnet. Einzelne WG-Zimmer werden ausgeschlossen.

In [ ]:
bereinigt = rohdaten[
    (rohdaten["is_furnished"] == False) &
    (rohdaten["is_temporary"] == False) &
    (rohdaten["price_unit"] == "monthly") &
    (rohdaten["object_category"] == "APARTMENT") &
    (rohdaten["offer_type"] == "RENT") &
    (rohdaten["object_type"] != "SINGLE_ROOM")
].copy()

bereinigt["surface_living"] = pd.to_numeric(bereinigt["surface_living"], errors="coerce")
bereinigt["number_of_rooms"] = pd.to_numeric(bereinigt["number_of_rooms"], errors="coerce")

bereinigt = bereinigt[
    bereinigt["surface_living"].between(15, 300) &
    bereinigt["number_of_rooms"].notna()
].copy()

bereinigt["rent_gross"] = bereinigt["rent_gross"].fillna(
    bereinigt["rent_net"] + bereinigt["rent_charges"]
)
bereinigt = bereinigt[bereinigt["rent_gross"].between(500, 10000)].copy()
bereinigt["city"] = bereinigt["city"].replace("Zurich", "Zürich")

print("Bereinigte Wohnungen:", len(bereinigt))

## 5. Neun Spalten mit den Namen des Modells erstellen
Das Modell erwartet `id`, `municipality`, `rooms`, `surface_m2`, `year_built`, `lat`, `lon`, `rent_gross`, `url`. Ein fehlendes Baujahr ist erlaubt.

In [ ]:
spalten = [
    "pk", "city", "number_of_rooms", "surface_living", "year_built",
    "latitude", "longitude", "rent_gross", "url"
]

modell_tabelle = bereinigt[spalten].copy()
modell_tabelle = modell_tabelle.rename(columns={
    "pk": "id",
    "city": "municipality",
    "number_of_rooms": "rooms",
    "surface_living": "surface_m2",
    "latitude": "lat",
    "longitude": "lon"
})

modell_tabelle["url"] = "https://flatfox.ch" + modell_tabelle["url"]
modell_tabelle.to_csv("data/flatfox_zurich.csv", index=False)

print("Datei:", "data/flatfox_zurich.csv")
print("Zeilen und Spalten:", modell_tabelle.shape)
display(modell_tabelle.head())

## 6. SQLite-Datenbank erstellen und kontrollieren
Die Tabelle heißt `listings` und enthält dieselben neun Spalten wie die CSV-Datei.

In [ ]:
verbindung = sqlite3.connect("data/wohnungen.db")
modell_tabelle.to_sql("listings", verbindung, if_exists="replace", index=False)

anzahl = pd.read_sql("SELECT COUNT(*) FROM listings", verbindung)
print(anzahl)

verbindung.close()

## 7. Dateien herunterladen (nur Google Colab)
Die erzeugten Dateien liegen im Ordner `data` in Colab. Die folgenden Befehle laden CSV und SQLite-Datenbank auf deinen Computer herunter.

In [ ]:
from google.colab import files
files.download("data/flatfox_zurich.csv")
files.download("data/wohnungen.db")